In [ ]:
import pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Pinecone

c:\Genai\LLM Langchain project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_doc(directory):
    file_loader=PyPDFDirectoryLoader(directory)
    documents=file_loader.load()
    return documents

In [3]:
doc=read_doc('documents/')
len(doc)

56

In [4]:
#Dividing the docs into chunks

def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc=text_splitter.split_documents(docs)
    return docs

In [5]:
documents = chunk_data(docs=doc) 
len(documents)

56

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using HuggingFace embeddings (Free & Local)
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
embeddings


C:\Users\mahim\AppData\Local\Temp\ipykernel_17700\4048595946.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9572.22it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [8]:
vectors=embeddings.embed_query('how are you?')
len(vectors)

384

In [ ]:
import os
import pinecone
from langchain_community.vectorstores import Pinecone

# This patches the old LangChain code so it understands the new Pinecone!
pc = pinecone.Pinecone(api_key=os.environ['PINECONE_API_KEY'])
pinecone.Index = type(pc.Index("langchainvector"))
# ---------------

index_name = "langchainvector"

# Push the chunks to Pinecone!
index = Pinecone.from_documents(documents, embeddings, index_name=index_name)

In [11]:
#Cosine Similarity Retreive Results
def retrieve_results(query,k=2):
    matching_results=index.similarity_search(query,k=k)
    return matching_results

In [23]:
import os
# Notice how we use langchain_classic instead of langchain!
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_openai import ChatOpenAI

# 1. Initialize the OpenRouter LLM using the key from your .env file
llm = ChatOpenAI(
    api_key=os.environ.get('OPENROUTER_API_KEY').strip(), 
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free"
)

# 2. Create your chain
chain = load_qa_chain(llm, chain_type="stuff")


In [24]:
query = "What is the document about?"
docs = retrieve_results(query)

# Ask the AI!
response = chain.run(input_documents=docs, question=query)
print(response)

Based on the provided content, the document is about **"Applications of Large Language Models"**.

Specifically, it covers two main applications:

1. **Content Creation and Writing Assistance** - LLMs are used to help create content like articles, scripts, and poetry, as well as improve writing in terms of style, grammar, and coherence.

2. **Conversational AI and Chatbots** - LLMs power advanced chatbots that enable more natural, context-aware conversations for customer service and virtual assistance applications.

The document appears to be educational material from Great Learning.
